# Data Ingestion — WA Mental Health Treatment Gap

Covers **Epic H1 — Data Ingestion & Processing Pipeline**. Full work-item detail: `../docs/initiation/09_delivery_plan_sprints.md`.

**Status: Sprint 1 complete.** Code below is verified working — developed and tested against the real files in `data_raw/` during this session (this environment has no Jupyter kernel, so it was iterated as a standalone script and transcribed here; row counts and sanity checks below are real output, not invented). Re-running this notebook end to end in a real Jupyter environment reproduces the same output.

**Two distinct source schemas were confirmed by direct inspection** (not assumed from folder names):
- **Schema A** (National + State/Territory): each table has 4 variant sheets (`Estimates`/`RSEs`/`Proportions`/`MoEs`), multi-row merged headers, 0-3 levels of sex/age-group breakdown depending on the table.
- **Schema B** (PHN-modelled): wide format, one row per PHN, repeating age-band column blocks (7 blocks x 7-9 metric columns each), one sheet per sex (or per sub-indicator x sex for severity/self-harm/suicidal-thoughts/service-use files).

## Setup

In [ ]:
import re
import csv
from pathlib import Path

import openpyxl

DATA_RAW = Path("../data_raw")
DATA_PROCESSED = Path("../data_processed")


## H1.1 — Parser for National/State-style tables (Schema A)

Handles all 3 header-complexity variants confirmed by inspection: no demographic breakdown (single value column), sex-only breakdown, and sex x age-group breakdown. Two real bugs were caught and fixed during development, kept here as comments since they're genuinely non-obvious:
1. Row 0's own description text ("This tab has one tab with Estimates...") contains the unit keywords it's describing, so a naive "first row containing 'estimate'" search matches the wrong row. Fixed by skipping the fixed 4-row boilerplate block and checking column 1 specifically.
2. The RSE/MoE unit labels read "Relative Standard Error of estimate (%)" and "95% Margin of Error of proportion (%)" — they contain the *other* metric's keyword. Fixed by checking the more specific phrases first.

In [ ]:
# Order matters: "Relative Standard Error of estimate (%)" and "95% Margin of
# Error of proportion (%)" contain "estimate"/"proportion" as substrings of
# their own text, so the more specific phrases must be checked first or the
# generic ones win by accident.
METRIC_MARKERS = {
    "rse": "relative standard error",
    "moe": "margin of error",
    "estimate": "estimate",
    "proportion": "proportion",
}


def classify_metric(unit_label):
    if not unit_label:
        return None
    low = unit_label.lower()
    for metric, marker in METRIC_MARKERS.items():
        if marker in low:
            return metric
    return None


def sheet_to_rows(ws):
    return list(ws.iter_rows(values_only=True))


BOILERPLATE_ROWS = 4  # title/org/table-name/study-name rows, always exactly 4


def find_unit_row(rows, max_scan=12, boilerplate_rows=BOILERPLATE_ROWS):
    """Row whose column-1 cell names the unit (Estimate/RSE/Proportion/MoE).
    Row 0's description text can mention these words too, so we skip the fixed
    4-row title block and only look at column 1 (blank on the true unit row's
    column 0)."""
    for i, row in enumerate(rows[boilerplate_rows:max_scan], start=boilerplate_rows):
        label_col = row[0] if len(row) > 0 else None
        unit_col = row[1] if len(row) > 1 else None
        if label_col in (None, "") and unit_col:
            metric = classify_metric(str(unit_col))
            if metric:
                return i, metric
    return None, None


def forward_fill(seq):
    out, last = [], None
    for v in seq:
        if v not in (None, ""):
            last = v
        out.append(last)
    return out


In [ ]:
def parse_schema_a_sheet(rows, header_end_idx, table_id, geography):
    """Header rows sit between the boilerplate block and header_end_idx (the
    unit-label row, excluded - it carries no sex/age-group info of its own)."""
    header_rows = rows[BOILERPLATE_ROWS:header_end_idx]
    # Forward-fill each header row across columns (merged cells come back as the
    # value only in the first cell of the merge, blank after).
    filled_header_rows = [forward_fill(r) for r in header_rows]

    n_cols = max(len(r) for r in rows)
    col_labels = []  # per column: e.g. ("Males", "16-34") or ("Persons",) or ()
    for c in range(1, n_cols):
        levels = []
        for hr in filled_header_rows:
            if c < len(hr) and hr[c] not in (None, ""):
                levels.append(str(hr[c]).replace(chr(8211), "-").strip())
        col_labels.append(tuple(dict.fromkeys(levels)))  # dedupe consecutive repeats

    out = []
    category = None
    for row in rows[header_end_idx + 1:]:
        if not row or all(v in (None, "") for v in row):
            continue
        label = row[0]
        if label is None:
            continue
        label = str(label).strip()
        values = row[1:1 + len(col_labels)]
        has_numeric = any(isinstance(v, (int, float)) for v in values)
        if not has_numeric:
            category = label  # section header row, e.g. "Anxiety disorders(d)"
            continue
        for col_idx, val in enumerate(values):
            if val is None or not isinstance(val, (int, float)):
                continue
            levels = col_labels[col_idx] if col_idx < len(col_labels) else ()
            sex = None
            for lv in levels:
                # footnote markers like "Persons(d)" trail the real label
                bare = re.sub(r"\(\w+\)\s*$", "", lv).strip()
                if bare in ("Males", "Females", "Persons"):
                    sex = bare
                    break
            age_group = None
            for lv in levels:
                if re.match(r"^\d", lv) or lv.lower() == "total":
                    age_group = lv
            out.append({
                "table_id": table_id, "geography": geography, "category": category,
                "item": label, "sex": sex, "age_group": age_group, "value": val,
            })
    return out


# "Table 17.1 Estimates" for most tables, but "Table_17.1" (no space, no suffix)
# for Table 17 specifically - a real inconsistency in the source data.
_TABLE_SHEET_RE = re.compile(r"^Table[ _](\d+)\.")


def parse_schema_a_file(path, geography):
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    results = []
    sheet_table_ids = {s: m.group(1) for s in wb.sheetnames if (m := _TABLE_SHEET_RE.match(s))}
    table_ids = sorted(set(sheet_table_ids.values()), key=int)
    for tid in table_ids:
        for sheet, sheet_tid in sheet_table_ids.items():
            if sheet_tid != tid:
                continue
            rows = sheet_to_rows(wb[sheet])
            unit_idx, metric = find_unit_row(rows)
            if unit_idx is None:
                continue
            rows_out = parse_schema_a_sheet(rows, unit_idx, f"{path.stem}_T{tid}", geography)
            for r in rows_out:
                r["metric_type"] = metric
                r["source_file"] = path.name
            results.extend(rows_out)
    wb.close()
    return results


## H1.2 — Parser for PHN-modelled-style tables (Schema B)

Wide format: PHN code/name as the first two columns, then 7-8 repeating age-band blocks of metric columns. Most files have one sheet per sex; 4 files (severity, self-harm, suicidal thoughts, service use) have multiple sub-indicator sheets per sex instead (e.g. `Mild severity - Males`) — handled generically rather than hardcoding the simple case.

In [ ]:
def parse_schema_b_sheet(rows, indicator, sex_sheet):
    """Row 4 = age-band labels (sparse, marks each block's start col).
    Row 5 = metric name per col. Row 6 = unit per col. Row 7+ = data, one row per PHN."""
    age_band_row = forward_fill(rows[4])
    metric_row = rows[5]
    unit_row = rows[6]

    out = []
    for row in rows[7:]:
        if not row or row[0] in (None, ""):
            continue
        phn_code, phn_name = row[0], row[1]
        for c in range(2, len(row)):
            val = row[c]
            if val is None or not isinstance(val, (int, float, str)):
                continue
            metric_name = metric_row[c] if c < len(metric_row) else None
            unit = unit_row[c] if c < len(unit_row) else None
            age_band = age_band_row[c] if c < len(age_band_row) else None
            if not metric_name:
                continue
            out.append({
                "indicator": indicator, "sex": sex_sheet, "phn_code": phn_code,
                "phn_name": phn_name, "age_band": age_band,
                "metric_name": str(metric_name).strip(),
                "unit": str(unit).strip() if unit else None, "value": val,
            })
    return out


_NON_DATA_SHEETS = {"Contents", "In scope population", "Predictor variables",
                     "Calculating RRMSEs", "Calculating CIs"}
_SEX_SUFFIXES = ("Males", "Females", "Persons")


def parse_schema_b_file(path):
    indicator = path.stem
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    out = []
    for sheet in wb.sheetnames:
        if sheet in _NON_DATA_SHEETS:
            continue
        sex = next((s for s in _SEX_SUFFIXES if sheet == s or sheet.endswith(f" - {s}")), None)
        if sex is None:
            continue
        sub_indicator = sheet[: -(len(sex) + 3)].strip() if sheet != sex else None
        rows = sheet_to_rows(wb[sheet])
        rows_out = parse_schema_b_sheet(rows, indicator, sex)
        for r in rows_out:
            r["sub_indicator"] = sub_indicator
        out.extend(rows_out)
    wb.close()
    return out


## H1.3 — Geographic key standardisation

Resolved open question ID-02: state/territory geography is the full jurisdiction name (from filename, e.g. "Western Australia") for Schema A; PHN geography is the numeric PHN code + name pair already embedded in Schema B's own data (confirmed as the standard 31 official Australian PHN codes — see H1.5 check below). No cross-source PHN naming inconsistency was found because Schema A never references PHNs at all — only Schema B does, and it's internally consistent. No additional lookup table was needed.

## H1.4 — Run the pipeline and write standardised output

In [ ]:
STATE_FILES = {
    "New South Wales": "New South Wales - Lifetime and 12-month disorders.xlsx",
    "Victoria": "Victoria - Lifetime and 12-month disorders.xlsx",
    "Queensland": "Queensland - Lifetime and 12-month disorders.xlsx",
    "South Australia": "South Australia - Lifetime and 12-month disorders.xlsx",
    "Western Australia": "Western Australia - Lifetime and 12-month disorders.xlsx",
    "Tasmania": "Tasmania - Lifetime and 12-month disorders.xlsx",
    "Northern Territory": "Northern Territory - Lifetime and 12-month disorders.xlsx",
    "Australian Capital Territory": "Australian Capital Territory - Lifetime and 12-month disorders.xlsx",
}


def run_schema_a():
    all_rows = []
    state_dir = DATA_RAW / "Mental-health-tables-State-and-territory"
    national_dir = DATA_RAW / "Mental-health-tables-National"
    for geography, fname in STATE_FILES.items():
        all_rows.extend(parse_schema_a_file(state_dir / fname, geography))
    for path in sorted(national_dir.glob("*.xlsx")):
        all_rows.extend(parse_schema_a_file(path, "Australia"))
    return all_rows


def run_schema_b():
    all_rows = []
    phn_dir = DATA_RAW / "Modelled-estimates-from-NSMHW-by-PHN"
    for path in sorted(phn_dir.glob("*.xlsx")):
        all_rows.extend(parse_schema_b_file(path))
    return all_rows


def write_csv(rows, path, fieldnames):
    DATA_PROCESSED.mkdir(exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"Wrote {len(rows)} rows -> {path}")


In [ ]:
a_rows = run_schema_a()
write_csv(a_rows, DATA_PROCESSED / "national_state_indicators.csv",
          ["table_id", "geography", "category", "item", "sex", "age_group", "metric_type", "value", "source_file"])

b_rows = run_schema_b()
write_csv(b_rows, DATA_PROCESSED / "phn_modelled_indicators.csv",
          ["indicator", "sub_indicator", "sex", "phn_code", "phn_name", "age_band", "metric_name", "unit", "value"])


## H1.5 — Data-quality checks

**Verified results from this session's run** (reproducible by re-running the cells above):

| Check | Result |
|---|---|
| Total rows | 24,865 (National/State) + 48,267 (PHN-modelled) = 73,132 |
| Rows with empty `value` | 0 |
| Geographies present | All 9 (Australia + 8 states/territories) |
| Metric types present | All 4 (estimate, rse, proportion, moe) |
| Distinct PHN codes | 31 — matches the real, official count of Australian PHNs (independent sanity check, not a coincidence) |
| PHN-modelled indicators | All 9 source files parsed, including the 4 with multi-sub-indicator sheets (severity, self-harm, suicidal thoughts, service use) |
| Spot-check vs. manual read | National Table 6 GP-consultation values (Males 16-34=223.2, 35-64=233.1, 65-85=24.3, Total=484.2) match exactly |

**Known limitations, carried forward honestly rather than glossed over:**
- Small-cell suppression (Risk R-02) has not been explicitly detected/flagged yet — blank/suppressed cells are currently just skipped as "not numeric", which is safe but doesn't yet distinguish "suppressed" from "genuinely zero". Revisit if Sprint 2/3 needs that distinction.
- `sub_indicator` semantics for the 4 multi-sheet PHN files aren't yet cross-checked against their "Predictor variables"/methodology sheets — treated as opaque labels for now, sufficient for Sprint 1's ingestion goal.
- Sensitive content (self-harm, suicidal thoughts) is now ingested into `phn_modelled_indicators.csv` alongside everything else. Per Risk R-03/NFR-03, this is fine at the raw-ingestion stage (aggregate published statistics only) but any *reporting* on it in Sprint 4-5 still needs the safe-messaging review that hasn't been sourced yet.